# Ollama セットアップ確認

Docker + Ollama の動作確認ノートブック。

**前提:** `src/serving/ollama/` で `make up && make pull` が完了していること。

In [ ]:
import requests
import base64
import json

OLLAMA_BASE_URL = "http://localhost:11434"
MODEL = "gemma4:26b"

## 1. 接続確認

In [ ]:
resp = requests.get(f"{OLLAMA_BASE_URL}/api/tags")
resp.raise_for_status()
models = resp.json().get("models", [])
print("利用可能なモデル:")
for m in models:
    print(f"  {m['name']} ({m['size'] / 1e9:.1f} GB)")

## 2. テキスト推論

In [ ]:
payload = {
    "model": MODEL,
    "prompt": "日本語で一言：今日の天気は？",
    "stream": False
}
resp = requests.post(f"{OLLAMA_BASE_URL}/api/generate", json=payload)
resp.raise_for_status()
print(resp.json()["response"])

## 3. 画像入力テスト

技術図面のサンプル画像を使って推論を確認する。

In [ ]:
IMAGE_PATH = "../../data/samples/sample_drawing.png"  # サンプル画像のパス

with open(IMAGE_PATH, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

payload = {
    "model": MODEL,
    "prompt": "この技術図面にコネクタや接続部品が含まれていますか？あれば位置と種類を説明してください。",
    "images": [image_b64],
    "stream": False
}
resp = requests.post(f"{OLLAMA_BASE_URL}/api/generate", json=payload)
resp.raise_for_status()
print(resp.json()["response"])

## 4. OpenAI SDK 互換 API テスト

In [ ]:
# pip install openai
from openai import OpenAI

client = OpenAI(
    base_url=f"{OLLAMA_BASE_URL}/v1",
    api_key="ollama",  # Ollamaはキー不要だが引数として必要
)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "OpenAI互換APIのテストです。一言で応答してください。"}
    ]
)
print(response.choices[0].message.content)